# mounding_lower_L3_K -> specs, pond as LAK

Import the run, rebuild the specs, drop `rch_pond`, add a LAK with a volumetric inflow
hydrograph. Same LAK workflow as the SW models.

In [ ]:
import myflopy as mf
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np

CRS = 2927
ROOT = Path(r'/home/lukem/models/mf6/tentrails')
DATA = Path(r'/home/lukem/models/mf6/Ten Trails/data')
RUN  = ROOT / 'runs' / 'mounding_lower_L3_K'

MODEL_NAME = 'tentrails'
SIM_NAME   = 'mounding_lak'

# ---- new pond -----------------------------------------------------------------
POND_GPKG     = DATA / ''          # <- pond footprint polygon
POND_CONTOURS = DATA / ''          # <- pond bathymetry contours
HYDROGRAPHS   = DATA / ''          # <- inflow hydrograph, volumetric rate
POND_LAYER    = 'pond'
STARTING_STAGE = 0.0               # <- pond invert / initial stage
BED_LEAKANCE   = 100

## Load

In [ ]:
run = mf.load_mf6_run(RUN, crs=f'EPSG:{CRS}')
run.load_all()
gwf = run.gwf
vor = run.vor

print(run.package_names)
print(vor.ncpl, 'cells', vor.crs)

## Grid / context

In [ ]:
disv_pkg = gwf.get_package('DISV')
top = disv_pkg.top.array
botm = disv_pkg.botm.array
idomain = disv_pkg.idomain.array
nlay = disv_pkg.nlay.get_data()

# Cut the pond into the loaded layering. There is no LayerStack here, so the arrays go
# into a LayerBuildResult and THAT is excavated -- a new result; `loaded` is unchanged.
# Without the cut MF6 counts each lake floor connection from its cell's TOP, not the bed.
lak_bathy = mf.Contours(contours=POND_CONTOURS, region_vector=POND_GPKG, resolution=1)
loaded = mf.LayerBuildResult(
    top=top, botm=botm, idomain=idomain,
    thickness=np.vstack([top[None], botm[:-1]]) - botm,
    names=[f'layer_{k + 1}' for k in range(nlay)],
    min_thickness=[0.0] * nlay, pinch=['floor'] * nlay,
    length_units='feet', time_units='days', vor=vor,
)
pond = loaded.excavate(bottom=lak_bathy, within=POND_GPKG, layer=POND_LAYER, name='pond')
print(pond.excavation_report().T)

vor.gdf_topbtm = None
# surfaces=pond is what carries the excavation's record to mf.lak
ctx = mf.ModelContext(grid=vor, domain=pond.idomain, surfaces=pond)

## Specs

In [ ]:
disv = mf.disv(
    nlay=nlay,
    ncpl=disv_pkg.ncpl.get_data(),
    nvert=disv_pkg.nvert.get_data(),
    vertices=disv_pkg.vertices.get_data().tolist(),
    cell2d=disv_pkg.cell2d.get_data().tolist(),
    top=pond.top, botm=pond.botm, idomain=pond.idomain,   # the excavated arrays
)

npf_pkg = gwf.get_package('NPF')
npf = mf.npf(k=npf_pkg.k.array, k33=npf_pkg.k33.array,
             icelltype=npf_pkg.icelltype.array, save_specific_discharge=True)

sto_pkg = gwf.get_package('STO')
sto = mf.sto(ss=sto_pkg.ss.array, sy=sto_pkg.sy.array,
             iconvert=sto_pkg.iconvert.array,
             steady_state={0: True}, transient={1: True})

ic = mf.ic(strt=gwf.get_package('IC').strt.array)

oc = mf.oc(head_filerecord=f'{MODEL_NAME}.hds',
           budget_filerecord=f'{MODEL_NAME}.cbb',
           saverecord=[('HEAD', 'LAST'), ('BUDGET', 'LAST')])

In [ ]:
def spd(name):
    return {p: gwf.get_package(name).stress_period_data.get_data(p)
            for p in gwf.get_package(name).stress_period_data.get_data()}

chd   = mf.chd(stress_period_data=spd('CHD'),   boundnames=True, name='chd')
drn   = mf.drn(stress_period_data=spd('DRN'),   boundnames=True, name='drn')
ghb   = mf.ghb(stress_period_data=spd('GHB'),   boundnames=True, name='ghb')
riv   = mf.riv(stress_period_data=spd('RIV'),   boundnames=True, name='riv')
crisp = mf.riv(stress_period_data=spd('CRISP'), boundnames=True, name='crisp')

for n, s in (('chd', chd), ('drn', drn), ('ghb', ghb), ('riv', riv), ('crisp', crisp)):
    print(n, len(s.options['stress_period_data'][0]))

In [ ]:
rch_pkg = gwf.get_package('RCH')
recharge = rch_pkg.recharge.get_data()
# irch from the excavated domain: the loaded IRCH still names layer 1 in the pond's dug
# cells, which the build refuses (ledger 321)
rch = mf.rch.array(context=ctx, recharge=recharge, irch='top_active', name='rch')

nper = len(recharge)
print('nper', nper)

`rch_pond` is read but not registered — the LAK replaces it.

In [ ]:
pond_pkg = gwf.get_package('RCH_POND')
pond_recharge = {p: a for p, a in pond_pkg.recharge.get_data().items() if a is not None}

pond_cells = np.flatnonzero(np.max(list(pond_recharge.values()), axis=0) > 0)
peak = max(float(np.max(a)) for a in pond_recharge.values())

print(len(pond_cells), 'pond cells |', len(pond_recharge), 'periods with data | peak', peak)

## Project

In [ ]:
proj = mf.Project(ROOT, name='tentrails')

proj.add_package('dis/L3K',   disv,  overwrite=True)
proj.add_package('npf/L3K',   npf,   overwrite=True)
proj.add_package('sto/L3K',   sto,   overwrite=True)
proj.add_package('ic/L3K',    ic,    overwrite=True)
proj.add_package('oc/L3K',    oc,    overwrite=True)
proj.add_package('chd/L3K',   chd,   overwrite=True)
proj.add_package('drn/L3K',   drn,   overwrite=True)
proj.add_package('ghb/L3K',   ghb,   overwrite=True)
proj.add_package('riv/L3K',   riv,   overwrite=True)
proj.add_package('crisp/L3K', crisp, overwrite=True)
proj.add_package('rch/L3K',   rch,   overwrite=True)

proj.packages.summary()

## LAK

Placeholders. Same shape as `lak/sw_flow`. The pond was cut into the layering in the
Grid / context cell, so `mf.lak` takes its bed from the excavation (no `lake_bottom`),
connects its floor at the bed and its walls into the neighbouring cells;
`unexcavated='raise'` refuses a lake the excavation's record did not reach.
`implicit=True` writes MF6's LAK `IMPLICIT` (an MF6 develop / 6.8 build; the COMPLEX
IMS below is BICGSTAB, which it needs).

In [ ]:
inflow = (pd.read_excel(HYDROGRAPHS).flow).to_list()      # <- volumetric rate, ft3/d

lak = mf.lak(
    nper=nper,
    context=ctx,
    lakes=POND_GPKG,
    layer=POND_LAYER,
    lake_id_field='name',
    starting_stage=STARTING_STAGE,
    connection_modes='bathy',
    bed_leakance=BED_LEAKANCE,
    inflow=inflow,
    unexcavated='raise',
    implicit=True,
)
print(lak.metadata['lake_rules'])                          # {'<name>': 'excavated'}

proj.add_package('lak/L3K', lak, overwrite=True)

## tdis / simulation

In [ ]:
tdis_pkg = run.sim.get_package('tdis')
perioddata = [tuple(r) for r in tdis_pkg.perioddata.get_data()]

tdis = mf.tdis(nper=nper, perioddata=perioddata, time_units='days',
               start_date_time='2017-09-30',
               ats=True, ats_dtfailadj=5.0, ats_dtadj=2.0)

model = mf.gwf(
    MODEL_NAME,
    context=ctx,
    **run.model_options,   # NEWTON etc. -- the model's own options, ledger 286
    packages=[
        mf.ref('dis/L3K'), mf.ref('npf/L3K'), mf.ref('sto/L3K'), mf.ref('ic/L3K'),
        mf.ref('chd/L3K'), mf.ref('ghb/L3K'), mf.ref('drn/L3K'),
        mf.ref('riv/L3K'), mf.ref('crisp/L3K'),
        mf.ref('rch/L3K'), mf.ref('lak/L3K'),
        mf.ref('oc/L3K'),
    ],
)

sim = mf.simulation(model, name=SIM_NAME,
                    packages=[tdis, mf.ims(complexity='COMPLEX')])
proj.add_simulation(sim, overwrite=True)

## Run

In [ ]:
new = proj.prepare_run('mounding_lak', SIM_NAME)
new.build()
new.built.built_model(MODEL_NAME).model.get_package_list()

In [ ]:
pd.DataFrame(new.resolution)

Specs rebuilt from a run hold typed cellids with no pin, so they read `unverifiable` —
fine on this grid, refused on another. Re-declare from the `.gpkg` sources if you move it.

In [ ]:
# new.execute()